# 07 — Inventory Optimization

Safety stock is derived from the spread of **lead-time forecast error**, not from historical demand variability. That is the entire reason forecasting has value: a better forecast shrinks the error spread and therefore the stock required. Sizing from demand variability returns the same answer no matter how good the forecast is.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


In [2]:
from src.inventory.policy import PolicyParams, z_for_service_level
p = PolicyParams.from_config(cfg)
print(f'lead time {p.lead_time_days} d + review {p.review_period_days} d = {p.protection_days} d of exposure')
print('z(95%) =', round(z_for_service_level(0.95), 4))

lead time 7 d + review 7 d = 14 d of exposure
z(95%) = 1.6449


Under periodic review the exposure is lead time **plus** the review period: a stockout can occur any time before the *next* order arrives. Omitting the review period systematically under-protects.

## Lead-time forecast error and safety stock

In [3]:
import pandas as pd
from src.inventory.policy import lead_time_forecast_errors, error_spread, build_policy
proc = cfg.resolve('data','processed_dir')
ew = pd.read_parquet(proc / 'error_windows.parquet')
lt = lead_time_forecast_errors(ew, 'lightgbm', p.protection_days)
sp = error_spread(lt)
print('windows per series (median):', int(sp.n_windows.median()))
sp[['error_std','error_q90','error_q95']].describe().round(3)

windows per series (median): 60


,error_std,error_q90,error_q95
count,600.000,600.000,600.000
mean,7.055,9.967,11.913
std,12.072,22.746,25.996
min,0.156,-4.148,-2.614
25%,2.255,2.413,3.184
50%,3.614,4.869,5.960
75%,6.997,9.577,11.344
max,159.648,405.291,453.588


In [4]:
test_preds = pd.read_parquet(proc / 'test_predictions.parquet')
pol = build_policy(cfg, test_preds, 'lightgbm', service_level=0.95, lt_errors=lt)
pol[['series_id','expected_lt_demand','error_std','safety_stock_normal',
     'safety_stock_empirical','reorder_point_normal']].head(10).round(2)

,series_id,expected_lt_demand,error_std,safety_stock_normal,safety_stock_empirical,reorder_point_normal
0,FOODS_1_004_CA_1,50.90,60.90,100.17,41.58,152.0
1,FOODS_1_004_TX_1,63.62,108.54,178.54,160.98,243.0
2,FOODS_1_004_WI_1,68.47,88.43,145.45,138.75,214.0
3,FOODS_1_031_CA_1,21.13,5.40,8.88,7.82,31.0
4,FOODS_1_031_TX_1,26.34,6.35,10.45,12.86,37.0
5,FOODS_1_031_WI_1,25.36,4.33,7.12,7.87,33.0
6,FOODS_1_038_CA_1,10.04,3.61,5.94,7.46,16.0
7,FOODS_1_038_TX_1,8.47,3.69,6.06,5.85,15.0
8,FOODS_1_038_WI_1,6.81,3.22,5.29,7.24,13.0
9,FOODS_1_073_CA_1,14.61,6.25,10.28,9.32,25.0


## Simulate: does the policy deliver its service level?

A reorder point from a formula is a claim. Replaying the held-out demand day by day is the test. Unmet demand is treated as **lost sales**, not backorders.

In [5]:
from src.inventory.simulate import simulate_policy, compare_policies
from src.inventory.policy import baseline_policy
from src.data.loader import test_fold
tf = test_fold(cfg)
base = baseline_policy(cfg, panel[panel.date <= tf.origin], 0.95)
sims = [simulate_policy(panel, base, tf.start, tf.end, p.lead_time_days,
                        p.review_period_days, 'reorder_point_normal',
                        policy_name='baseline_demand_variability'),
        simulate_policy(panel, pol, tf.start, tf.end, p.lead_time_days,
                        p.review_period_days, 'reorder_point_normal',
                        policy_name='forecast_normal_ss')]
compare_policies(sims).round(4)

,policy,series,total_demand,units_short,fill_rate,cycle_service_level,stockout_day_rate,avg_inventory_units,orders_placed,series_with_stockout
0,baseline_demand_variability,600,24754.0,675.7778,0.9727,0.9742,0.0081,15650.0560,917.0,47
1,forecast_normal_ss,600,24754.0,526.6775,0.9787,0.9825,0.0047,17430.6796,870.0,32


## Cost the policies

> Only unit value comes from the data. Margin, holding rate, ordering cost and the stockout penalty are declared assumptions.

In [6]:
from src.inventory import costs as C
uv = C.unit_values(panel, as_of=tf.origin)
econ = C.Economics.from_config(cfg)
pd.DataFrame([{'policy': s.policy_name,
               **C.cost_summary(C.cost_simulation(s.per_series, uv, econ, days=28))}
              for s in sims]).round(2)

,policy,holding_cost_cu,stockout_cost_cu,ordering_cost_cu,total_cost_cu,units_short,fill_rate,avg_inventory_units,cost_per_unit_demand_cu
0,baseline_demand_variability,1040.11,626.83,1834.0,3500.94,675.78,0.97,15650.06,0.14
1,forecast_normal_ss,1155.86,336.03,1740.0,3231.88,526.68,0.98,17430.68,0.13
